# Veri Madenciliği Dönem Projesi — İkinci El SUV Fiyat Sınıflandırma

**Tek dosya versiyonu.** Tüm adımlar bu notebook'ta.

## Kullanım
1. Bu notebook'u, ham Excel dosyası ile AYNI klasöre koy
2. Excel dosyası adı: `arabam_ham_veri.xlsx` olmalı (değiştirirsen aşağıda EXCEL_PATH'i güncelle)
3. Kernel → Restart & Run All

## İçerik
1. Veri yükleme
2. Keşif (EDA)
3. Ön işleme
4. Train/test split
5. 8 algoritma — her biri ayrı hücrede
6. Sonuç karşılaştırma

## 0. Ayarlar

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 60)

# ========================================
# AYARLAR — BURAYI DEĞİŞTİR
# ========================================
EXCEL_PATH = 'data/arabam_ham_veri.xlsx'  # data/ klasöründen oku
SHEET_NAME = 'ARABAMVS'              # Kullanılacak sayfa
RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f'Mevcut klasör: {os.getcwd()}')
print(f'Excel dosyası var mı: {os.path.exists(EXCEL_PATH)}')

## 1. Veri yükleme

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
print(f'Boyut: {df.shape[0]} satır × {df.shape[1]} sütun')
df.head()

## 2. Keşif (EDA)

In [ ]:
# Eksik veri oranları
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]
print(f'Eksik değeri olan {len(missing)} sütun var.')
print(missing.head(15))

In [ ]:
# Marka dağılımı
print('Marka dağılımı:')
print(df['Marka'].value_counts())
print('\nYakıt tipi:')
print(df['Yakıt Tipi'].value_counts())

## 3. Ön işleme

In [ ]:
# 3.1 — Fiyatı temizle (string → sayı)
def clean_price(series):
    if series.dtype == 'object' or str(series.dtype) == 'str':
        cleaned = (series.astype(str)
                         .str.replace('TL', '', regex=False)
                         .str.replace('.', '', regex=False)
                         .str.replace(',', '.', regex=False)
                         .str.replace(' ', '', regex=False)
                         .str.strip())
        return pd.to_numeric(cleaned, errors='coerce')
    return series

df['Fiyat'] = clean_price(df['Fiyat'])
df = df.dropna(subset=['Fiyat'])
print(f'Fiyat istatistikleri:')
print(df['Fiyat'].describe())

In [ ]:
# 3.2 — Fiyat sınıfı oluştur (4 quartile)
df['Fiyat_Sinifi'] = pd.qcut(df['Fiyat'], q=4, labels=['Ekonomik', 'Orta', 'Yuksek', 'Premium'])
print('Sınıf dağılımı:')
print(df['Fiyat_Sinifi'].value_counts())

# Grafik
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['Fiyat'].plot(kind='hist', bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Fiyat dağılımı')
df['Fiyat_Sinifi'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Fiyat Sınıfı dağılımı')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 — Gereksiz sütunları at
DROP_COLS = [
    '_Başlık', '_URL', 'Kopyalandı', 'İlan Tarihi',
    'Şanzıman', 'Ağır Hasarlı', 'Ağırlık', 'Plaka Uyruğu',
    'Motor ve Performans', 'Boyut ve Kapasite', 'Yakıt Tüketimi',
    'Ön Lastik', '@dropdown', 'Model', 'Üretim Yılı (İlk/Son)',
    'Ort. Yakıt Tüketimi',
]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print(f'Kalan sütun: {df.shape[1]}')

In [ ]:
# 3.4 — Numerik sütunlardaki birimleri sök ('305.000 km' → 305000)
NUMERIC_COLS = [
    'Kilometre', 'Motor Hacmi', 'Motor Gücü', 'Yakıt Deposu',
    'Tork', 'Maksimum Güç', 'Minimum Güç', 'Hızlanma (0-100)',
    'Maksimum Hız', 'Ortalama Yakıt Tüketimi',
    'Şehir İçi Yakıt Tüketimi', 'Şehir Dışı Yakıt Tüketimi',
    'Uzunluk', 'Genişlik', 'Yükseklik', 'Boş Ağırlığı',
    'Bagaj Hacmi',
]

for col in NUMERIC_COLS:
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        cleaned = df[col].astype(str).str.extract(r'([\d.,]+)', expand=False)
        cleaned = cleaned.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(cleaned, errors='coerce')

# Kasko / Sigorta
for col in ['Ortalama Kasko', 'Ortalama Trafik Sigortası']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Sütun tipleri temizlendi.')
print(df.dtypes.value_counts())

In [ ]:
# 3.5 — Boolean sütunlar
BOOL_COLS = ['Orjinal', 'Lokal boyalı', 'Boyalı', 'Değişmiş', 'Belirtilmemiş', 'Boya-değişen']
for col in BOOL_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
print('Boolean sütunlar temizlendi.')

In [ ]:
# 3.6 — Kategorik sütunlar için one-hot encoding
CATEGORICAL_COLS = ['Marka', 'Seri', 'Vites Tipi', 'Yakıt Tipi',
                    'Kasa Tipi', 'Renk', 'Çekiş', 'Kimden', 'Sınıfı']
categorical_present = [c for c in CATEGORICAL_COLS if c in df.columns]

# Eksikleri doldur
for col in categorical_present:
    df[col] = df[col].fillna(df[col].mode()[0])

# One-hot
df_encoded = pd.get_dummies(df, columns=categorical_present, drop_first=True)
print(f'One-hot sonrası boyut: {df_encoded.shape}')

In [ ]:
# 3.7 — Eksik numerik değerleri medyan ile doldur
numeric_present = df_encoded.select_dtypes(include=[np.number]).columns
df_encoded[numeric_present] = df_encoded[numeric_present].fillna(df_encoded[numeric_present].median())

print(f'Toplam eksik hücre: {df_encoded.isnull().sum().sum()}')

## 4. Train / Test Split + Ölçekleme

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Hedef ve feature'ları ayır
y = df_encoded['Fiyat_Sinifi']
X = df_encoded.drop(columns=['Fiyat', 'Fiyat_Sinifi'])

# Sadece numerik sütunları al (one-hot bool'ları da sayısal)
X = X.select_dtypes(include=[np.number, 'bool']).astype(float)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Ölçekle
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')
print(f'Sınıflar:')
print(y_train.value_counts())

## 5. Değerlendirme fonksiyonu

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

results = []

def evaluate(model, X_tr, X_te, y_tr, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    res = {
        'model': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_te, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_te, y_pred, average='macro', zero_division=0),
    }
    results.append(res)
    
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'Accuracy : {res["accuracy"]:.4f}')
    print(f'F1 macro : {res["f1_macro"]:.4f}')
    print(f'Precision: {res["precision"]:.4f}')
    print(f'Recall   : {res["recall"]:.4f}')
    return model

## 6. Algoritmalar — her hücre bir ekip üyesinin algoritması

Her üye kendi hücresini alıp optimize edebilir.

### 6.1 — k-NN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = evaluate(
    KNeighborsClassifier(n_neighbors=5),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'k-NN (k=5)'
)

### 6.2 — ID3 (Entropy tabanlı Karar Ağacı)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_id3 = evaluate(
    DecisionTreeClassifier(criterion='entropy', random_state=RANDOM_STATE),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'ID3 (entropy)'
)

### 6.3 — C4.5 (Entropy + budanmış)

In [ ]:
model_c45 = evaluate(
    DecisionTreeClassifier(criterion='entropy', max_depth=10,
                           min_samples_split=10, random_state=RANDOM_STATE),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'C4.5 (budanmış)'
)

### 6.4 — CART (Gini tabanlı)

In [ ]:
model_cart = evaluate(
    DecisionTreeClassifier(criterion='gini', random_state=RANDOM_STATE),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'CART (gini)'
)

### 6.5 — Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

model_nb = evaluate(
    GaussianNB(),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'Naive Bayes (Gaussian)'
)

### 6.6 — Lojistik Regresyon

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = evaluate(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'Lojistik Regresyon'
)

### 6.7 — K-Means (kümelerle sınıflandırma)

In [ ]:
from sklearn.cluster import KMeans
from scipy.stats import mode

# 4 küme bul, her kümeye çoğunluk sınıfı ata
km = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
train_clusters = km.fit_predict(X_train_scaled)

# Her kümenin çoğunluk sınıfını bul
cluster_to_label = {}
for c in range(4):
    mask = train_clusters == c
    if mask.sum() > 0:
        cluster_to_label[c] = y_train[mask].mode()[0]

# Test tahmini
test_clusters = km.predict(X_test_scaled)
y_pred_km = [cluster_to_label[c] for c in test_clusters]

res = {
    'model': 'K-Means (4 küme)',
    'accuracy': accuracy_score(y_test, y_pred_km),
    'precision': precision_score(y_test, y_pred_km, average='macro', zero_division=0),
    'recall': recall_score(y_test, y_pred_km, average='macro', zero_division=0),
    'f1_macro': f1_score(y_test, y_pred_km, average='macro', zero_division=0),
}
results.append(res)
print(f'\nK-Means → Accuracy: {res["accuracy"]:.4f}, F1: {res["f1_macro"]:.4f}')

### 6.8 — Random Forest (bonus)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = evaluate(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    X_train_scaled, X_test_scaled, y_train, y_test,
    'Random Forest (200 ağaç)'
)

## 7. SONUÇ KARŞILAŞTIRMA

In [ ]:
results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
print('\n🏆 TÜM MODELLER — F1 macro sıralaması:')
print(results_df.to_string(index=False))

# Grafik
fig, ax = plt.subplots(figsize=(12, 6))
results_df.set_index('model')[['accuracy', 'precision', 'recall', 'f1_macro']].plot(
    kind='bar', ax=ax, colormap='viridis'
)
plt.title('Tüm modeller — metrik karşılaştırma')
plt.ylabel('Skor')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('outputs/model_karsilastirma.png', dpi=150)
plt.show()

print(f'\n🥇 KAZANAN: {results_df.iloc[0]["model"]} (F1={results_df.iloc[0]["f1_macro"]:.4f})')

## 8. Sonuçları kaydet

In [ ]:
results_df.to_csv('outputs/sonuclar.csv', index=False)
df_encoded.to_csv('outputs/arabam_temiz.csv', index=False)

print('✓ Kaydedildi:')
print('  - outputs/sonuclar.csv (model karşılaştırma)')
print('  - outputs/arabam_temiz.csv (temizlenmiş veri)')
print('  - outputs/model_karsilastirma.png (grafik)')


---

## Sırada ne var?

Buradan sonra ekip olarak yapılacaklar:

1. **Her üye bir algoritmayı sahiplenir** — yukarıdaki hücrelerden birini alır
2. **GridSearchCV ile hyperparameter tuning** yapar (her algoritma için farklı parametreler)
3. **Özellik mühendisliği** dener (yaş = 2026-Yıl, km/yıl oranı, vs.)
4. **Confusion matrix** çizer, zayıflıkları analiz eder
5. **Raporda** her adımı detayla

Başarılar! 🚗